In [ ]:
import pandas as pd
from transformers import BertTokenizer, BertForSequenceClassification
import torch
import re

# Load data
df = pd.read_csv('../data/test.csv')
texts = df['text'].tolist()

def clean_text(s: str):
    s = re.sub(r'[“”]', '"', s)
     
    # Replace all curly single quotes with '
    s = re.sub(r"[‘’]", "'", s)
    
    # Replace en dash and em dash with hyphen
    s = re.sub(r"[–—]", "-", s)

    # Only retain alphanumeric, whitespace characters, single and double quotes, and hyphens
    s = re.sub(pattern=rf"[^a-zA-Z0-9\s\-\'\"]", repl="", string=s, flags=re.IGNORECASE)

    # Remove extra whitespaces
    s = re.sub(pattern=r"\s+", repl=" ", string=s).strip()

    return s

def preprocess(text: str):
    return clean_text(text)

# Load pretrained model and tokenizer
tokenizer = BertTokenizer.from_pretrained("../models/bert")
model = BertForSequenceClassification.from_pretrained("../models/bert")
model.eval()

# Tokenize
cleaned_text = [preprocess(text) for text in texts]
inputs = tokenizer(cleaned_text, padding=True, truncation=True, return_tensors='pt')

# Predict
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1).tolist()

# Save
df['predictions'] = predictions
df.to_csv('../predictions/bert_predictions.csv', index=False)